# Parse Gemini Responses

This notebook parses the responses from the Gemini model and creates a dataset with the generated reports.

In [ ]:
import pandas as pd

train_dataset = pd.read_csv("train_dataset.csv")
train_dataset["generated_response"] = ""
with open("results.txt", "r") as f:
    lines = f.readlines()

In [ ]:
current_index = 0
current_response = []

for line in lines:
    if line.startswith("Index"):
        # The index written in the file has 10 added. Oops
        current_index = int(line.split(":")[0].split(" ")[1]) - 10
        current_response = []
    elif line.startswith("End of index"):
        reports = "\n".join(current_response).split("### Poročilo")
        
        # Sometimes the first report starts with ### Poročilo, sometimes it doesn't. Ignore the first one if it's empty. 
        if len(reports) == 11 and reports[0] == "":
            reports = reports[1:]
            
        if len(reports) == 11 and reports[0] != "":
            temp = 0
        
        if len(reports) != 10:
            print(f"Index {current_index} has {len(reports)} reports, expected 10.")
            continue
            
        for i in range(10):
            train_dataset.at[current_index + i, "generated_response"] = reports[i].strip()
    else:
        current_response.append(line)
        
train_dataset.to_csv("train_dataset_generated_reports.csv", index=False)
        

In [ ]:
num_no_response = sum(train_dataset["generated_response"] == "")
print(f"Number of rows with no response: {num_no_response}")

In [ ]:
train_dataset_clean = train_dataset[train_dataset["generated_response"] != ""]
train_dataset.to_csv("train_dataset_generated_reports.csv", index=False)